# Experimento 2 — Inducción no supervisada de fragmentos relacionales

## Objetivo

El objetivo de este experimento es explorar un enfoque no supervisado para detectar patrones relacionales recurrentes en textos históricos del corpus DICAT.

A diferencia del experimento anterior, centrado en el clustering de verbos y tripletas sintácticas, en este caso se trabajará con fragmentos relacionales completos enriquecidos con contexto semántico.

La hipótesis de partida es que las relaciones ontológicas no se expresan únicamente mediante verbos aislados, sino mediante construcciones lingüísticas más complejas que incluyen:

- contexto discursivo,
- modalidad,
- evidencia documental,
- negación,
- temporalidad,
- y tipos de entidades.

Por ello, el experimento buscará inducir agrupaciones semánticas de fragmentos relacionales utilizando embeddings contextualizados y técnicas de clustering no supervisado.

## Pipeline general

1. Carga y preparación del corpus.
2. Segmentación en frases.
3. Extracción automática de entidades y fragmentos relacionales.
4. Construcción de embeddings semánticos.
5. Clustering no supervisado de fragmentos.
6. Análisis e interpretación de los clusters generados.

In [1]:
# ============================================================
# INICIALIZACIÓN
# ============================================================

DATASET_PATH = "../Data/DicatJuanRana_Dataset.csv"
TEXT_COLUMN = "clarified_sentences"

SPACY_MODEL = "es_core_news_lg"
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"


In [2]:
# ============================================================
# IMPORTS
# ============================================================

import re
import warnings

import pandas as pd
import numpy as np

import spacy

from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

# ============================================================
# CARGA DE MODELOS
# ============================================================

print("Cargando modelo spaCy...")
nlp = spacy.load(SPACY_MODEL)

print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print("Modelos cargados correctamente.")

Cargando modelo spaCy...
Cargando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelos cargados correctamente.


In [3]:
# ============================================================
# CARGA DEL DATASET
# ============================================================

df = pd.read_csv(DATASET_PATH, sep=";")

print(f"Número de registros: {len(df)}")
print(f"Columnas disponibles:\n{df.columns.tolist()}")

df.head()

Número de registros: 38
Columnas disponibles:
['year', 'text', 'sentences', 'clarified_sentences']


,year,text,sentences,clarified_sentences
0,0,"Aunque su verdadero nombre era Cosme Pérez, fu...","[""Aunque su verdadero nombre era Cosme Pérez, ...","[""Aunque su verdadero nombre era Cosme Pérez, ..."
1,1617,"Según Cotarelo , a quien sigue Sáez Raposo , C...","['Según Cotarelo, a quien sigue Sáez Raposo, ...","['Según Cotarelo, a quien sigue Sáez Raposo, S..."
2,1621,Consta la lista de la compañía de Juan Bautist...,['Consta la lista de la compañía de Juan Bauti...,['Consta la lista de la compañía de Juan Bauti...
3,1624,Cosme Pérez aparece en la nómina de la compañí...,['Cosme Pérez aparece en la nómina de la compa...,['Cosme Pérez aparece en la nómina de la compa...
4,1634,"Según Cotarelo, Cosme Pérez pertenecía a la co...","['Según Cotarelo, Cosme Pérez pertenecía a la ...","['Según Cotarelo, Cosme Pérez pertenecía a la ..."


In [4]:
# ============================================================
# LIMPIEZA BÁSICA DEL TEXTO
# ============================================================

import ast

def clean_text(text):
    if pd.isna(text):
        return ""

    # Si viene como lista guardada en texto: "['frase 1', 'frase 2']"
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            text = " ".join(parsed)
    except:
        text = str(text)

    # Eliminar marcas HTML
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<.*?>", " ", text)

    # Normalizar espacios
    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["clean_text"] = df[TEXT_COLUMN].apply(clean_text)

df[["clean_text"]].head()

,clean_text
0,"Aunque su verdadero nombre era Cosme Pérez, Co..."
1,"Según Cotarelo, a quien sigue Sáez Raposo, Sáe..."
2,Consta la lista de la compañía de Juan Bautist...
3,Cosme Pérez aparece en la nómina de la compañí...
4,"Según Cotarelo, Cosme Pérez pertenecía a la co..."


In [5]:
# ============================================================
# PROCESAMIENTO CON SPACY
# ============================================================

print("Procesando documentos con spaCy...")

df["doc"] = df["clean_text"].apply(nlp)

print("Procesamiento completado.")

df[["clean_text"]].head()

Procesando documentos con spaCy...
Procesamiento completado.


,clean_text
0,"Aunque su verdadero nombre era Cosme Pérez, Co..."
1,"Según Cotarelo, a quien sigue Sáez Raposo, Sáe..."
2,Consta la lista de la compañía de Juan Bautist...
3,Cosme Pérez aparece en la nómina de la compañí...
4,"Según Cotarelo, Cosme Pérez pertenecía a la co..."


In [6]:
# ============================================================
# EXTRACCIÓN DE FRASES
# ============================================================

sentences_data = []

for doc_id, row in df.iterrows():
    doc = row["doc"]
    
    for sent_id, sent in enumerate(doc.sents):
        sent_text = sent.text.strip()
        
        if sent_text:
            sentences_data.append({
                "document_id": doc_id,
                "sentence_id": sent_id,
                "sentence": sent_text
            })

df_sentences = pd.DataFrame(sentences_data)

print(f"Número total de frases extraídas: {len(df_sentences)}")

df_sentences.head(10)

Número total de frases extraídas: 345


,document_id,sentence_id,sentence
0,0,0,"Aunque su verdadero nombre era Cosme Pérez, Co..."
1,0,1,Aunque Cotarelo aventuró que pudo haber nacido...
2,0,2,Cosme Pérez fue hijo de Damián Pérez e Isabel ...
3,0,3,"Según algunas fuentes, como Sánchez Arjona y S..."
4,0,4,"Por su lado, Rennert puso en duda el matrimoni..."
5,0,5,"Sin embargo, ambos matrimonios no se encuentra..."
6,0,6,"De hecho, la Genealogía tan sólo documenta el ..."
7,0,7,En la misma Genealogía se hace alusión a la en...
8,0,8,"Como muestra de su popularidad, el autor de la..."
9,0,9,"Según la misma fuente, Cosme Pérez gozó de la ..."


In [7]:
# ============================================================
# INSPECCIÓN FRASE A FRASE
# ============================================================

def inspect_sentence(i):
    sent = df_sentences.loc[i, "sentence"]
    doc = nlp(sent)

    print("="*100)
    print(f"FRASE {i}")
    print("="*100)
    print(sent)

    print("\nENTIDADES:")
    for ent in doc.ents:
        print(f"- {ent.text} ({ent.label_})")

    print("\nTOKENS RELEVANTES:")
    for token in doc:
        if token.pos_ in ["VERB", "AUX", "NOUN", "PROPN", "ADJ"]:
            print(
                f"{token.text:20} lemma={token.lemma_:20} "
                f"pos={token.pos_:8} dep={token.dep_:12} head={token.head.text}"
            )

In [8]:
inspect_sentence(0)

FRASE 0
Aunque su verdadero nombre era Cosme Pérez, Cosme Pérez fue conocido artísticamente como 'Juan Rana', según documenta ya la Genealogía —este apodo ya se le aplicaba en 1636—.

ENTIDADES:
- Cosme Pérez (PER)
- Cosme Pérez (PER)
- Juan Rana (PER)
- Genealogía —este apodo (MISC)

TOKENS RELEVANTES:
verdadero            lemma=verdadero            pos=ADJ      dep=amod         head=nombre
nombre               lemma=nombre               pos=NOUN     dep=nsubj        head=Cosme
era                  lemma=ser                  pos=AUX      dep=cop          head=Cosme
Cosme                lemma=Cosme                pos=PROPN    dep=obl          head=conocido
Pérez                lemma=Pérez                pos=PROPN    dep=flat         head=Cosme
Cosme                lemma=Cosme                pos=PROPN    dep=conj         head=Cosme
Pérez                lemma=Pérez                pos=PROPN    dep=flat         head=Cosme
fue                  lemma=ser                  pos=AUX      dep=aux

In [9]:
# ============================================================
# EXTRACCIÓN DE FRAGMENTOS RELACIONALES POR VENTANA
# ============================================================

def extract_relational_fragments(sentence, window=8):
    doc = nlp(sentence)
    fragments = []

    for token in doc:
        if token.pos_ == "VERB":
            start = max(token.i - window, 0)
            end = min(token.i + window + 1, len(doc))

            fragment = doc[start:end].text

            fragments.append({
                "trigger": token.text,
                "lemma": token.lemma_,
                "dependency": token.dep_,
                "fragment": fragment
            })

    return fragments

In [10]:
fragments = extract_relational_fragments(df_sentences.loc[0, "sentence"])

for f in fragments:
    print("="*80)
    print(f)

{'trigger': 'conocido', 'lemma': 'conocer', 'dependency': 'ROOT', 'fragment': "nombre era Cosme Pérez, Cosme Pérez fue conocido artísticamente como 'Juan Rana', según"}
{'trigger': 'documenta', 'lemma': 'documentar', 'dependency': 'advcl', 'fragment': "artísticamente como 'Juan Rana', según documenta ya la Genealogía —este apodo ya se"}
{'trigger': 'aplicaba', 'lemma': 'aplicar', 'dependency': 'advcl', 'fragment': 'la Genealogía —este apodo ya se le aplicaba en 1636—.'}


In [11]:
def clean_fragment(fragment):
    fragment = re.sub(r"[—]+", " ", fragment)
    fragment = re.sub(r"\s+", " ", fragment)
    return fragment.strip(" ,.;:-")


def extract_relational_fragments(sentence, window=8):
    doc = nlp(sentence)
    fragments = []

    for token in doc:
        if token.pos_ == "VERB":
            start = max(token.i - window, 0)
            end = min(token.i + window + 1, len(doc))

            fragment = clean_fragment(doc[start:end].text)

            fragments.append({
                "trigger": token.text,
                "lemma": token.lemma_,
                "dependency": token.dep_,
                "fragment": fragment
            })

    return fragments

In [12]:
fragments = extract_relational_fragments(df_sentences.loc[0, "sentence"], window=8)

for f in fragments:
    print("="*80)
    print(f)

{'trigger': 'conocido', 'lemma': 'conocer', 'dependency': 'ROOT', 'fragment': "nombre era Cosme Pérez, Cosme Pérez fue conocido artísticamente como 'Juan Rana', según"}
{'trigger': 'documenta', 'lemma': 'documentar', 'dependency': 'advcl', 'fragment': "artísticamente como 'Juan Rana', según documenta ya la Genealogía este apodo ya se"}
{'trigger': 'aplicaba', 'lemma': 'aplicar', 'dependency': 'advcl', 'fragment': 'la Genealogía este apodo ya se le aplicaba en 1636'}


In [13]:
# ============================================================
# INSPECCIÓN DETALLADA DE UNA FRASE
# ============================================================

sentence = df_sentences.loc[0, "sentence"]
doc = nlp(sentence)

print(sentence)

for token in doc:
    if token.pos_ == "VERB":
        print("\n" + "="*80)
        print(f"TRIGGER: {token.text}")
        print(f"LEMMA: {token.lemma_}")
        print(f"DEP: {token.dep_}")
        print(f"HEAD: {token.head.text}")

        print("\nHIJOS:")
        for child in token.children:
            print(f"- {child.text:20} dep={child.dep_:12} pos={child.pos_:8}")

        print("\nFRAGMENTO VENTANA:")
        start = max(token.i - 8, 0)
        end = min(token.i + 9, len(doc))
        print(doc[start:end].text)

        print("\nSUBTREE:")
        print(" ".join([t.text for t in token.subtree]))

Aunque su verdadero nombre era Cosme Pérez, Cosme Pérez fue conocido artísticamente como 'Juan Rana', según documenta ya la Genealogía —este apodo ya se le aplicaba en 1636—.

TRIGGER: conocido
LEMMA: conocer
DEP: ROOT
HEAD: conocido

HIJOS:
- Cosme                dep=obl          pos=PROPN   
- fue                  dep=aux          pos=AUX     
- artísticamente       dep=advmod       pos=ADV     
- Juan                 dep=obj          pos=PROPN   
- aplicaba             dep=advcl        pos=VERB    
- —                    dep=punct        pos=PUNCT   
- .                    dep=punct        pos=PUNCT   

FRAGMENTO VENTANA:
nombre era Cosme Pérez, Cosme Pérez fue conocido artísticamente como 'Juan Rana', según

SUBTREE:
Aunque su verdadero nombre era Cosme Pérez , Cosme Pérez fue conocido artísticamente como ' Juan Rana ' , según documenta ya la Genealogía — este apodo ya se le aplicaba en 1636 — .

TRIGGER: documenta
LEMMA: documentar
DEP: advcl
HEAD: aplicaba

HIJOS:
- ,            

In [112]:
# ============================================================
# FUNCIONES DE EXTRACCIÓN DE FRAGMENTOS
# ============================================================

def clean_fragment(fragment):

    fragment = re.sub(r"[—]+", " ", fragment)
    fragment = re.sub(r"\s+", " ", fragment)

    return fragment.strip(" ,.;:-")


# ------------------------------------------------------------
# VENTANA SIMPLE
# ------------------------------------------------------------

def get_window(doc, token, window=8):

    start = max(token.i - window, 0)
    end = min(token.i + window + 1, len(doc))

    return clean_fragment(doc[start:end].text)


# ------------------------------------------------------------
# SUBTREE SINTÁCTICO
# ------------------------------------------------------------

def get_subtree(token):

    return clean_fragment(
        " ".join([t.text for t in token.subtree])
    )


# ------------------------------------------------------------
# FRAGMENTO BASADO EN EVENTOS
# ------------------------------------------------------------

def get_event_window(doc, token, left_window=6, right_window=12):

    # Buscar siguiente verbo
    next_verbs = [
        t.i for t in doc
        if t.pos_ == "VERB" and t.i > token.i
    ]

    # Cortar antes del siguiente verbo
    if next_verbs:
        next_verb = min(next_verbs)
        end = min(next_verb, token.i + right_window + 1)
    else:
        end = min(token.i + right_window + 1, len(doc))

    start = max(token.i - left_window, 0)

    fragment = doc[start:end].text

    return clean_fragment(fragment)


# ------------------------------------------------------------
# EXTRACCIÓN COMPLETA
# ------------------------------------------------------------

def extract_candidate_fragments(sentence, window=8):

    doc = nlp(sentence)

    rows = []

    for token in doc:

        if token.pos_ == "VERB":

            rows.append({
                "trigger": token.text,
                "lemma": token.lemma_,
                "dep": token.dep_,
                "head": token.head.text,
                "fragment_window": get_window(doc, token, window),
                "fragment_subtree": get_subtree(token),
                "fragment_event": get_event_window(doc, token),
                "sentence": sentence
            })

    return rows

In [113]:
candidates = extract_candidate_fragments(df_sentences.loc[0, "sentence"], window=8)

pd.DataFrame(candidates)

,trigger,lemma,dep,head,fragment_window,fragment_subtree,fragment_event,sentence
0,conocido,conocer,ROOT,conocido,"nombre era Cosme Pérez, Cosme Pérez fue conoci...","Aunque su verdadero nombre era Cosme Pérez , C...","Cosme Pérez, Cosme Pérez fue conocido artístic...","Aunque su verdadero nombre era Cosme Pérez, Co..."
1,documenta,documentar,advcl,aplicaba,"artísticamente como 'Juan Rana', según documen...",según documenta ya la Genealogía,"'Juan Rana', según documenta ya la Genealogía ...","Aunque su verdadero nombre era Cosme Pérez, Co..."
2,aplicaba,aplicar,advcl,conocido,la Genealogía este apodo ya se le aplicaba en ...,según documenta ya la Genealogía este apodo ya...,este apodo ya se le aplicaba en 1636,"Aunque su verdadero nombre era Cosme Pérez, Co..."


In [114]:
# ============================================================
# SELECCIÓN AUTOMÁTICA DEL FRAGMENTO MÁS ÚTIL
# ============================================================

def extract_candidate_fragments(sentence, window=8):
    doc = nlp(sentence)
    rows = []

    for token in doc:
        if token.pos_ == "VERB":
            rows.append({
                "trigger": token.text,
                "lemma": token.lemma_,
                "dep": token.dep_,
                "head": token.head.text,
                "fragment_window": get_window(doc, token, window),
                "fragment_subtree": get_subtree(token),
                "fragment_event": get_event_window(doc, token),
                "sentence": sentence
            })

    return rows

In [115]:
# ============================================================
# SELECCIÓN AUTOMÁTICA DEL FRAGMENTO FINAL
# ============================================================

def select_fragment(row):

    subtree = str(row["fragment_subtree"])
    window = str(row["fragment_window"])

    sentence_len = len(str(row["sentence"]).split())
    subtree_len = len(subtree.split())

    # Si el subtree no ocupa casi toda la frase,
    # suele ser más limpio que la ventana
    if subtree_len <= sentence_len * 0.6:
        return subtree

    return window

In [116]:
df_candidates = pd.DataFrame(candidates)

df_candidates["fragment_final"] = df_candidates.apply(
    select_fragment,
    axis=1
)

df_candidates[[
    "trigger",
    "lemma",
    "dep",
    "fragment_window",
    "fragment_subtree",
    "fragment_final"
]]

,trigger,lemma,dep,fragment_window,fragment_subtree,fragment_final
0,conocido,conocer,ROOT,"nombre era Cosme Pérez, Cosme Pérez fue conoci...","Aunque su verdadero nombre era Cosme Pérez , C...","nombre era Cosme Pérez, Cosme Pérez fue conoci..."
1,documenta,documentar,advcl,"artísticamente como 'Juan Rana', según documen...",según documenta ya la Genealogía,según documenta ya la Genealogía
2,aplicaba,aplicar,advcl,la Genealogía este apodo ya se le aplicaba en ...,según documenta ya la Genealogía este apodo ya...,según documenta ya la Genealogía este apodo ya...


In [117]:
# ============================================================
# PRUEBA CONTROLADA - FRASE 1
# ============================================================

sentence_index = 1

candidates_1 = extract_candidate_fragments(
    df_sentences.loc[sentence_index, "sentence"],
    window=8
)

df_candidates_1 = pd.DataFrame(candidates_1)

df_candidates_1["fragment_final"] = df_candidates_1.apply(
    select_fragment,
    axis=1
)

print(df_sentences.loc[sentence_index, "sentence"])

df_candidates_1[[
    "trigger",
    "lemma",
    "dep",
    "fragment_window",
    "fragment_subtree",
    "fragment_final"
]]

Aunque Cotarelo aventuró que pudo haber nacido en Madrid, hoy sabemos que Cosme Pérez nació en Tudela de Duero (Valladolid), en cuya iglesia parroquial Cosme Pérez fue bautizado el 7 de abril de 1593.


,trigger,lemma,dep,fragment_window,fragment_subtree,fragment_final
0,aventuró,aventurar,advcl,Aunque Cotarelo aventuró que pudo haber nacido...,Aunque Cotarelo aventuró que pudo haber nacido...,Aunque Cotarelo aventuró que pudo haber nacido...
1,nacido,nacer,ccomp,Aunque Cotarelo aventuró que pudo haber nacido...,que pudo haber nacido en Madrid,que pudo haber nacido en Madrid
2,sabemos,saber,ROOT,"que pudo haber nacido en Madrid, hoy sabemos q...",Aunque Cotarelo aventuró que pudo haber nacido...,"que pudo haber nacido en Madrid, hoy sabemos q..."
3,nació,nacer,ccomp,"en Madrid, hoy sabemos que Cosme Pérez nació e...",que Cosme Pérez nació en Tudela de Duero ( Val...,"en Madrid, hoy sabemos que Cosme Pérez nació e..."
4,bautizado,bautizar,advcl,en cuya iglesia parroquial Cosme Pérez fue bau...,en cuya iglesia parroquial Cosme Pérez fue bau...,en cuya iglesia parroquial Cosme Pérez fue bau...


In [118]:
# ============================================================
# PRUEBA CONTROLADA - FRASE 4
# ============================================================

sentence_index = 4

candidates_4 = extract_candidate_fragments(
    df_sentences.loc[sentence_index, "sentence"],
    window=8
)

df_candidates_4 = pd.DataFrame(candidates_4)

df_candidates_4["fragment_final"] = df_candidates_4.apply(
    select_fragment,
    axis=1
)

print(df_sentences.loc[sentence_index, "sentence"])

df_candidates_4[[
    "trigger",
    "lemma",
    "dep",
    "fragment_window",
    "fragment_subtree",
    "fragment_final"
]]

Por su lado, Rennert puso en duda el matrimonio de Cosme Pérez con Bernarda Ramírez y creyó más probable que Cosme Pérez estuviese casado con Bernarda Manuela ['la Grifona'], que sería, según esto, la primera mujer de Cosme Pérez.


,trigger,lemma,dep,fragment_window,fragment_subtree,fragment_final
0,puso,poner,ROOT,"Por su lado, Rennert puso en duda el matrimoni...","Por su lado , Rennert puso en duda el matrimon...","Por su lado, Rennert puso en duda el matrimoni..."
1,creyó,creer,conj,matrimonio de Cosme Pérez con Bernarda Ramírez...,y creyó más probable que Cosme Pérez estuviese...,matrimonio de Cosme Pérez con Bernarda Ramírez...


In [119]:
# ============================================================
# SELECCIÓN ESTRUCTURAL AUTOMÁTICA
# ============================================================

def select_fragment_auto(row):

    subtree = str(row["fragment_subtree"])
    window = str(row["fragment_window"])

    subtree_len = len(subtree.split())
    window_len = len(window.split())

    # Ratio entre subtree y ventana
    ratio = subtree_len / max(window_len, 1)

    # Si el subtree es suficientemente compacto,
    # lo consideramos más informativo
    if ratio <= 0.75:
        return subtree

    return window

In [120]:
# ============================================================
# REPROBAR FRASE 4 CON SELECCIÓN AUTOMÁTICA
# ============================================================

df_candidates_4["fragment_auto"] = df_candidates_4.apply(
    select_fragment_auto,
    axis=1
)

df_candidates_4[[
    "trigger",
    "lemma",
    "fragment_subtree",
    "fragment_window",
    "fragment_auto"
]]

,trigger,lemma,fragment_subtree,fragment_window,fragment_auto
0,puso,poner,"Por su lado , Rennert puso en duda el matrimon...","Por su lado, Rennert puso en duda el matrimoni...","Por su lado, Rennert puso en duda el matrimoni..."
1,creyó,creer,y creyó más probable que Cosme Pérez estuviese...,matrimonio de Cosme Pérez con Bernarda Ramírez...,matrimonio de Cosme Pérez con Bernarda Ramírez...


In [121]:
# ============================================================
# COMPARACIÓN MANUAL DE REPRESENTACIONES
# ============================================================

def compare_representations(sentence_index):

    sentence = df_sentences.loc[sentence_index, "sentence"]

    print("="*120)
    print(f"FRASE {sentence_index}")
    print("="*120)
    print(sentence)

    candidates = extract_candidate_fragments(sentence)

    df_compare = pd.DataFrame(candidates)

    for _, row in df_compare.iterrows():

        print("\n" + "-"*120)

        print(f"TRIGGER: {row['trigger']}")
        print(f"LEMMA: {row['lemma']}")
        print(f"DEP: {row['dep']}")

        print("\nWINDOW:")
        print(row["fragment_window"])

        print("\nSUBTREE:")
        print(row["fragment_subtree"])

In [122]:
compare_representations(0)

FRASE 0
Aunque su verdadero nombre era Cosme Pérez, Cosme Pérez fue conocido artísticamente como 'Juan Rana', según documenta ya la Genealogía —este apodo ya se le aplicaba en 1636—.

------------------------------------------------------------------------------------------------------------------------
TRIGGER: conocido
LEMMA: conocer
DEP: ROOT

WINDOW:
nombre era Cosme Pérez, Cosme Pérez fue conocido artísticamente como 'Juan Rana', según

SUBTREE:
Aunque su verdadero nombre era Cosme Pérez , Cosme Pérez fue conocido artísticamente como ' Juan Rana ' , según documenta ya la Genealogía este apodo ya se le aplicaba en 1636

------------------------------------------------------------------------------------------------------------------------
TRIGGER: documenta
LEMMA: documentar
DEP: advcl

WINDOW:
artísticamente como 'Juan Rana', según documenta ya la Genealogía este apodo ya se

SUBTREE:
según documenta ya la Genealogía

---------------------------------------------------------------

In [123]:
compare_representations(1)

FRASE 1
Aunque Cotarelo aventuró que pudo haber nacido en Madrid, hoy sabemos que Cosme Pérez nació en Tudela de Duero (Valladolid), en cuya iglesia parroquial Cosme Pérez fue bautizado el 7 de abril de 1593.

------------------------------------------------------------------------------------------------------------------------
TRIGGER: aventuró
LEMMA: aventurar
DEP: advcl

WINDOW:
Aunque Cotarelo aventuró que pudo haber nacido en Madrid, hoy

SUBTREE:
Aunque Cotarelo aventuró que pudo haber nacido en Madrid

------------------------------------------------------------------------------------------------------------------------
TRIGGER: nacido
LEMMA: nacer
DEP: ccomp

WINDOW:
Aunque Cotarelo aventuró que pudo haber nacido en Madrid, hoy sabemos que Cosme Pérez

SUBTREE:
que pudo haber nacido en Madrid

------------------------------------------------------------------------------------------------------------------------
TRIGGER: sabemos
LEMMA: saber
DEP: ROOT

WINDOW:
que pudo haber 

In [124]:
compare_representations(4)

FRASE 4
Por su lado, Rennert puso en duda el matrimonio de Cosme Pérez con Bernarda Ramírez y creyó más probable que Cosme Pérez estuviese casado con Bernarda Manuela ['la Grifona'], que sería, según esto, la primera mujer de Cosme Pérez.

------------------------------------------------------------------------------------------------------------------------
TRIGGER: puso
LEMMA: poner
DEP: ROOT

WINDOW:
Por su lado, Rennert puso en duda el matrimonio de Cosme Pérez con

SUBTREE:
Por su lado , Rennert puso en duda el matrimonio de Cosme Pérez con Bernarda Ramírez y creyó más probable que Cosme Pérez estuviese casado con Bernarda Manuela [ ' la Grifona ' ] , que sería , según esto , la primera mujer de Cosme Pérez

------------------------------------------------------------------------------------------------------------------------
TRIGGER: creyó
LEMMA: creer
DEP: conj

WINDOW:
matrimonio de Cosme Pérez con Bernarda Ramírez y creyó más probable que Cosme Pérez estuviese casado con

SUB

In [125]:
# ============================================================
# MÉTRICAS AUTOMÁTICAS DE FRAGMENTOS
# ============================================================

def fragment_metrics(fragment):

    doc = nlp(fragment)

    return {
        "num_tokens": len(doc),
        "num_entities": len(doc.ents),
        "num_verbs": len([t for t in doc if t.pos_ == "VERB"]),
        "num_propn": len([t for t in doc if t.pos_ == "PROPN"])
    }

In [126]:
# ============================================================
# COMPARACIÓN AUTOMÁTICA DE REPRESENTACIONES
# ============================================================

for _, row in df_candidates_4.iterrows():

    print("\n" + "="*100)
    print(f"TRIGGER: {row['trigger']}")

    print("\nWINDOW")
    print(row["fragment_window"])
    print(fragment_metrics(row["fragment_window"]))

    print("\nSUBTREE")
    print(row["fragment_subtree"])
    print(fragment_metrics(row["fragment_subtree"]))


TRIGGER: puso

WINDOW
Por su lado, Rennert puso en duda el matrimonio de Cosme Pérez con
{'num_tokens': 14, 'num_entities': 2, 'num_verbs': 1, 'num_propn': 3}

SUBTREE
Por su lado , Rennert puso en duda el matrimonio de Cosme Pérez con Bernarda Ramírez y creyó más probable que Cosme Pérez estuviese casado con Bernarda Manuela [ ' la Grifona ' ] , que sería , según esto , la primera mujer de Cosme Pérez
{'num_tokens': 47, 'num_entities': 7, 'num_verbs': 2, 'num_propn': 13}

TRIGGER: creyó

WINDOW
matrimonio de Cosme Pérez con Bernarda Ramírez y creyó más probable que Cosme Pérez estuviese casado con
{'num_tokens': 17, 'num_entities': 3, 'num_verbs': 1, 'num_propn': 6}

SUBTREE
y creyó más probable que Cosme Pérez estuviese casado con Bernarda Manuela [ ' la Grifona ' ] , que sería , según esto , la primera mujer de Cosme Pérez
{'num_tokens': 31, 'num_entities': 4, 'num_verbs': 1, 'num_propn': 8}


In [127]:
# ============================================================
# SCORE AUTOMÁTICO DE FRAGMENTO
# ============================================================

def fragment_score(fragment):

    metrics = fragment_metrics(fragment)

    num_tokens = metrics["num_tokens"]
    num_entities = metrics["num_entities"]
    num_verbs = metrics["num_verbs"]

    # Evitar divisiones por cero
    num_tokens = max(num_tokens, 1)
    num_verbs = max(num_verbs, 1)

    score = num_entities / (num_verbs * num_tokens)

    return round(score, 4)

In [128]:
# ============================================================
# COMPARACIÓN DE SCORES
# ============================================================

for _, row in df_candidates_4.iterrows():

    window_score = fragment_score(row["fragment_window"])
    subtree_score = fragment_score(row["fragment_subtree"])

    print("\n" + "="*100)
    print(f"TRIGGER: {row['trigger']}")

    print("\nWINDOW")
    print(row["fragment_window"])
    print(f"Score: {window_score}")

    print("\nSUBTREE")
    print(row["fragment_subtree"])
    print(f"Score: {subtree_score}")


TRIGGER: puso

WINDOW
Por su lado, Rennert puso en duda el matrimonio de Cosme Pérez con
Score: 0.1429

SUBTREE
Por su lado , Rennert puso en duda el matrimonio de Cosme Pérez con Bernarda Ramírez y creyó más probable que Cosme Pérez estuviese casado con Bernarda Manuela [ ' la Grifona ' ] , que sería , según esto , la primera mujer de Cosme Pérez
Score: 0.0745

TRIGGER: creyó

WINDOW
matrimonio de Cosme Pérez con Bernarda Ramírez y creyó más probable que Cosme Pérez estuviese casado con
Score: 0.1765

SUBTREE
y creyó más probable que Cosme Pérez estuviese casado con Bernarda Manuela [ ' la Grifona ' ] , que sería , según esto , la primera mujer de Cosme Pérez
Score: 0.129


In [129]:
# ============================================================
# EMBEDDINGS DE FRAGMENTOS
# ============================================================

def get_embedding(text):

    embedding = embedding_model.encode(
        text,
        normalize_embeddings=True
    )

    return embedding

In [130]:
# ============================================================
# EMBEDDINGS PARA FRASE 0
# ============================================================

fragments_text = df_candidates["fragment_final"].tolist()

embeddings = embedding_model.encode(
    fragments_text,
    normalize_embeddings=True
)

print(f"Número de embeddings: {len(embeddings)}")
print(f"Dimensión embedding: {len(embeddings[0])}")

Número de embeddings: 3
Dimensión embedding: 384


In [131]:
# ============================================================
# MATRIZ DE SIMILITUD SEMÁNTICA
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=df_candidates["trigger"],
    columns=df_candidates["trigger"]
)

similarity_df

trigger,conocido,documenta,aplicaba
trigger,,,
conocido,1.000000,0.181476,0.191247
documenta,0.181476,1.000000,0.702790
aplicaba,0.191247,0.702790,1.000000


In [132]:
# ============================================================
# EMBEDDINGS Y SIMILITUD - FRASE 1
# ============================================================

fragments_text_1 = df_candidates_1["fragment_final"].tolist()

embeddings_1 = embedding_model.encode(
    fragments_text_1,
    normalize_embeddings=True
)

similarity_matrix_1 = cosine_similarity(embeddings_1)

similarity_df_1 = pd.DataFrame(
    similarity_matrix_1,
    index=df_candidates_1["trigger"],
    columns=df_candidates_1["trigger"]
)

similarity_df_1

trigger,aventuró,nacido,sabemos,nació,bautizado
trigger,,,,,
aventuró,1.000000,0.935967,0.802440,0.732002,0.341863
nacido,0.935967,1.000000,0.827652,0.727460,0.328524
sabemos,0.802440,0.827652,1.000000,0.961995,0.554874
nació,0.732002,0.727460,0.961995,1.000000,0.552813
bautizado,0.341863,0.328524,0.554874,0.552813,1.000000


In [133]:
# ============================================================
# MINI DATASET DE FRAGMENTOS
# ============================================================

all_fragments = []

for df_temp in [df_candidates, df_candidates_1, df_candidates_4]:

    for _, row in df_temp.iterrows():

        all_fragments.append({
            "trigger": row["trigger"],
            "lemma": row["lemma"],
            "fragment": row["fragment_final"]
        })

df_mini = pd.DataFrame(all_fragments)

df_mini

,trigger,lemma,fragment
0,conocido,conocer,"nombre era Cosme Pérez, Cosme Pérez fue conoci..."
1,documenta,documentar,según documenta ya la Genealogía
2,aplicaba,aplicar,según documenta ya la Genealogía este apodo ya...
3,aventuró,aventurar,Aunque Cotarelo aventuró que pudo haber nacido...
4,nacido,nacer,que pudo haber nacido en Madrid
5,sabemos,saber,"que pudo haber nacido en Madrid, hoy sabemos q..."
6,nació,nacer,"en Madrid, hoy sabemos que Cosme Pérez nació e..."
7,bautizado,bautizar,en cuya iglesia parroquial Cosme Pérez fue bau...
8,puso,poner,"Por su lado, Rennert puso en duda el matrimoni..."
9,creyó,creer,matrimonio de Cosme Pérez con Bernarda Ramírez...


In [166]:
# ============================================================
# CREAR REGISTRO ESTÁNDAR DEL POOL
# ============================================================

def build_pool_record(row, sentence_index, source_type="verbal"):

    return {
        "sentence_index": sentence_index,
        "trigger": row.get("trigger"),
        "lemma": row.get("lemma"),
        "syntactic_role": row.get("syntactic_role"),
        "fragment": row.get("fragment_final"),
        "fragment_no_entities": row.get("fragment_no_entities"),
        "source_type": source_type
    }

In [134]:
# ============================================================
# ESTADO INCREMENTAL INICIAL
# ============================================================

seen_entities = set()
seen_fragments = []

print("Estado incremental inicializado.")

Estado incremental inicializado.


In [135]:
# ============================================================
# LIMPIEZA AUTOMÁTICA DE ENTIDADES
# ============================================================

def clean_entity(entity):

    entity = str(entity)

    # Si spaCy une texto antes/después de raya larga, nos quedamos con la primera parte
    entity = entity.split("—")[0]

    # Limpieza básica
    entity = re.sub(r"\s+", " ", entity)

    return entity.strip(" ,.;:-")

In [136]:
# ============================================================
# PROCESAMIENTO INCREMENTAL DE UNA FRASE
# ============================================================

def process_incremental_sentence(sentence_index):

    sentence = df_sentences.loc[sentence_index, "sentence"]

    doc = nlp(sentence)

    # --------------------------------------------------------
    # ENTIDADES
    # --------------------------------------------------------

    entities = set([
        clean_entity(ent.text)
        for ent in doc.ents
        if clean_entity(ent.text)
    ])

    new_entities = entities - seen_entities

    # --------------------------------------------------------
    # FRAGMENTOS RELACIONALES
    # --------------------------------------------------------

    candidates = extract_candidate_fragments(
        sentence,
        window=8
    )

    df_cand = pd.DataFrame(candidates)

    if len(df_cand) > 0:

        df_cand["fragment_final"] = df_cand.apply(
            select_fragment,
            axis=1
        )

    # --------------------------------------------------------
    # RESULTADO
    # --------------------------------------------------------

    return {
        "sentence_index": sentence_index,
        "sentence": sentence,
        "entities": entities,
        "new_entities": new_entities,
        "candidates": df_cand
    }

In [137]:
# ============================================================
# PROCESAR FRASE 0
# ============================================================

result_0 = process_incremental_sentence(0)

print("="*120)
print("FRASE")
print("="*120)

print(result_0["sentence"])

print("\n" + "="*120)
print("ENTIDADES NUEVAS")
print("="*120)

print(result_0["new_entities"])

print("\n" + "="*120)
print("FRAGMENTOS")
print("="*120)

result_0["candidates"][[
    "trigger",
    "lemma",
    "fragment_final"
]]

FRASE
Aunque su verdadero nombre era Cosme Pérez, Cosme Pérez fue conocido artísticamente como 'Juan Rana', según documenta ya la Genealogía —este apodo ya se le aplicaba en 1636—.

ENTIDADES NUEVAS
{'Cosme Pérez', 'Juan Rana', 'Genealogía'}

FRAGMENTOS


,trigger,lemma,fragment_final
0,conocido,conocer,"nombre era Cosme Pérez, Cosme Pérez fue conoci..."
1,documenta,documentar,según documenta ya la Genealogía
2,aplicaba,aplicar,según documenta ya la Genealogía este apodo ya...


In [138]:
# ============================================================
# ACTUALIZAR ESTADO CON FRASE 0
# ============================================================

seen_entities.update(result_0["new_entities"])

for _, row in result_0["candidates"].iterrows():
    seen_fragments.append({
        "sentence_index": 0,
        "trigger": row["trigger"],
        "lemma": row["lemma"],
        "fragment": row["fragment_final"]
    })

print(f"Entidades vistas: {len(seen_entities)}")
print(f"Fragmentos vistos: {len(seen_fragments)}")

Entidades vistas: 3
Fragmentos vistos: 3


In [139]:
# ============================================================
# PROCESAR FRASE 1
# ============================================================

result_1 = process_incremental_sentence(1)

print("="*120)
print("FRASE")
print("="*120)
print(result_1["sentence"])

print("\n" + "="*120)
print("ENTIDADES NUEVAS")
print("="*120)
print(result_1["new_entities"])

print("\n" + "="*120)
print("FRAGMENTOS")
print("="*120)

result_1["candidates"][[
    "trigger",
    "lemma",
    "fragment_final"
]]

FRASE
Aunque Cotarelo aventuró que pudo haber nacido en Madrid, hoy sabemos que Cosme Pérez nació en Tudela de Duero (Valladolid), en cuya iglesia parroquial Cosme Pérez fue bautizado el 7 de abril de 1593.

ENTIDADES NUEVAS
{'Valladolid', 'Madrid', 'Cotarelo', 'Tudela de Duero'}

FRAGMENTOS


,trigger,lemma,fragment_final
0,aventuró,aventurar,Aunque Cotarelo aventuró que pudo haber nacido...
1,nacido,nacer,que pudo haber nacido en Madrid
2,sabemos,saber,"que pudo haber nacido en Madrid, hoy sabemos q..."
3,nació,nacer,"en Madrid, hoy sabemos que Cosme Pérez nació e..."
4,bautizado,bautizar,en cuya iglesia parroquial Cosme Pérez fue bau...


In [140]:
# ============================================================
# COMPARAR FRAGMENTOS NUEVOS CON FRAGMENTOS YA VISTOS
# ============================================================

previous_fragments = [item["fragment"] for item in seen_fragments]
new_fragments = result_1["candidates"]["fragment_final"].tolist()

all_texts = previous_fragments + new_fragments

emb = embedding_model.encode(
    all_texts,
    normalize_embeddings=True
)

previous_emb = emb[:len(previous_fragments)]
new_emb = emb[len(previous_fragments):]

similarities = cosine_similarity(new_emb, previous_emb)

df_similarity = pd.DataFrame(
    similarities,
    index=result_1["candidates"]["trigger"],
    columns=[item["trigger"] for item in seen_fragments]
)

df_similarity

,conocido,documenta,aplicaba
trigger,,,
aventuró,0.385579,0.417190,0.269234
nacido,0.374183,0.388421,0.267288
sabemos,0.678870,0.321790,0.281590
nació,0.669824,0.255004,0.226233
bautizado,0.566254,0.295514,0.336170


In [141]:
# ============================================================
# ELIMINAR ENTIDADES DE LOS FRAGMENTOS
# ============================================================

def remove_entities(text):

    doc = nlp(str(text))
    clean = str(text)

    for ent in doc.ents:
        clean = clean.replace(ent.text, f"[{ent.label_}]")

    return clean

In [142]:
# ============================================================
# CREAR FRAGMENTOS SIN ENTIDADES PARA FRASE 1
# ============================================================

result_1["candidates"] = result_1["candidates"].copy()

result_1["candidates"]["fragment_no_entities"] = result_1["candidates"]["fragment_final"].apply(
    remove_entities
)

result_1["candidates"][[
    "trigger",
    "fragment_final",
    "fragment_no_entities"
]]

,trigger,fragment_final,fragment_no_entities
0,aventuró,Aunque Cotarelo aventuró que pudo haber nacido...,Aunque [PER] aventuró que pudo haber nacido en...
1,nacido,que pudo haber nacido en Madrid,que pudo haber nacido en [LOC]
2,sabemos,"que pudo haber nacido en Madrid, hoy sabemos q...","que pudo haber nacido en [LOC], hoy sabemos qu..."
3,nació,"en Madrid, hoy sabemos que Cosme Pérez nació e...","en [LOC], hoy sabemos que [PER] nació en [LOC]..."
4,bautizado,en cuya iglesia parroquial Cosme Pérez fue bau...,en cuya iglesia parroquial [PER] fue bautizado...


In [143]:
# ============================================================
# ELIMINAR ENTIDADES DEL FRAGMENTO
# ============================================================

def remove_entities(text):
    doc = nlp(text)
    clean = text

    for ent in doc.ents:
        clean = clean.replace(ent.text, f"[{ent.label_}]")

    return clean

In [144]:
# ============================================================
# COMPARAR FRAGMENTOS SIN ENTIDADES
# ============================================================

previous_fragments_no_ent = [
    remove_entities(item["fragment"])
    for item in seen_fragments
]

new_fragments_no_ent = result_1["candidates"]["fragment_no_entities"].tolist()

all_texts_no_ent = previous_fragments_no_ent + new_fragments_no_ent

emb_no_ent = embedding_model.encode(
    all_texts_no_ent,
    normalize_embeddings=True
)

previous_emb_no_ent = emb_no_ent[:len(previous_fragments_no_ent)]
new_emb_no_ent = emb_no_ent[len(previous_fragments_no_ent):]

similarities_no_ent = cosine_similarity(
    new_emb_no_ent,
    previous_emb_no_ent
)

df_similarity_no_ent = pd.DataFrame(
    similarities_no_ent,
    index=result_1["candidates"]["trigger"],
    columns=[item["trigger"] for item in seen_fragments]
)

df_similarity_no_ent

,conocido,documenta,aplicaba
trigger,,,
aventuró,0.392481,0.434400,0.301527
nacido,0.383050,0.485688,0.331570
sabemos,0.450788,0.372876,0.279547
nació,0.483987,0.392534,0.251452
bautizado,0.222947,0.259927,0.385607


In [145]:
# ============================================================
# SIMILITUD INTERNA DE LA FRASE
# ============================================================

fragments_internal = result_1["candidates"]["fragment_no_entities"].tolist()

emb_internal = embedding_model.encode(
    fragments_internal,
    normalize_embeddings=True
)

sim_internal = cosine_similarity(emb_internal)

df_internal = pd.DataFrame(
    sim_internal,
    index=result_1["candidates"]["trigger"],
    columns=result_1["candidates"]["trigger"]
)

df_internal

trigger,aventuró,nacido,sabemos,nació,bautizado
trigger,,,,,
aventuró,1.000000,0.903905,0.789610,0.623013,0.275915
nacido,0.903905,1.000000,0.889851,0.714803,0.327497
sabemos,0.789610,0.889851,1.000000,0.898565,0.300761
nació,0.623013,0.714803,0.898565,1.000000,0.255499
bautizado,0.275915,0.327497,0.300761,0.255499,1.000000


In [146]:
# ============================================================
# DETECCIÓN DE REDUNDANCIA INTERNA
# ============================================================

threshold = 0.85

internal_pairs = []

triggers = result_1["candidates"]["trigger"].tolist()

for i in range(len(triggers)):
    for j in range(i + 1, len(triggers)):
        sim = sim_internal[i, j]

        if sim >= threshold:
            internal_pairs.append({
                "trigger_1": triggers[i],
                "trigger_2": triggers[j],
                "similarity": round(sim, 4)
            })

df_internal_pairs = pd.DataFrame(internal_pairs)

df_internal_pairs

,trigger_1,trigger_2,similarity
0,aventuró,nacido,0.9039
1,nacido,sabemos,0.8899
2,sabemos,nació,0.8986


In [147]:
# ============================================================
# AÑADIR ROL SINTÁCTICO AUTOMÁTICO
# ============================================================

def syntactic_role(dep):
    if dep == "ROOT":
        return "main"
    elif dep in ["ccomp", "xcomp"]:
        return "embedded_proposition"
    elif dep in ["advcl", "acl"]:
        return "subordinate"
    elif dep == "conj":
        return "coordinated"
    else:
        return "other"


result_1["candidates"]["syntactic_role"] = result_1["candidates"]["dep"].apply(
    syntactic_role
)

result_1["candidates"][[
    "trigger",
    "lemma",
    "dep",
    "syntactic_role",
    "fragment_no_entities"
]]

,trigger,lemma,dep,syntactic_role,fragment_no_entities
0,aventuró,aventurar,advcl,subordinate,Aunque [PER] aventuró que pudo haber nacido en...
1,nacido,nacer,ccomp,embedded_proposition,que pudo haber nacido en [LOC]
2,sabemos,saber,ROOT,main,"que pudo haber nacido en [LOC], hoy sabemos qu..."
3,nació,nacer,ccomp,embedded_proposition,"en [LOC], hoy sabemos que [PER] nació en [LOC]..."
4,bautizado,bautizar,advcl,subordinate,en cuya iglesia parroquial [PER] fue bautizado...


In [148]:
# ============================================================
# AGRUPAR POR ESTRUCTURA SINTÁCTICA + LEMA
# ============================================================

grouped_candidates = (
    result_1["candidates"]
    .groupby(["lemma", "syntactic_role"])
)

for group_name, group_df in grouped_candidates:

    print("\n" + "="*120)
    print(f"GRUPO: {group_name}")
    print("="*120)

    display(group_df[[
        "trigger",
        "fragment_no_entities"
    ]])


GRUPO: ('aventurar', 'subordinate')


,trigger,fragment_no_entities
0,aventuró,Aunque [PER] aventuró que pudo haber nacido en...



GRUPO: ('bautizar', 'subordinate')


,trigger,fragment_no_entities
4,bautizado,en cuya iglesia parroquial [PER] fue bautizado...



GRUPO: ('nacer', 'embedded_proposition')


,trigger,fragment_no_entities
1,nacido,que pudo haber nacido en [LOC]
3,nació,"en [LOC], hoy sabemos que [PER] nació en [LOC]..."



GRUPO: ('saber', 'main')


,trigger,fragment_no_entities
2,sabemos,"que pudo haber nacido en [LOC], hoy sabemos qu..."


In [149]:
# ============================================================
# SIMILITUD DENTRO DE CADA GRUPO ESTRUCTURAL
# ============================================================

for group_name, group_df in grouped_candidates:

    if len(group_df) < 2:
        continue

    print("\n" + "="*120)
    print(f"GRUPO: {group_name}")
    print("="*120)

    texts = group_df["fragment_no_entities"].tolist()

    emb_group = embedding_model.encode(
        texts,
        normalize_embeddings=True
    )

    sim_group = cosine_similarity(emb_group)

    df_sim_group = pd.DataFrame(
        sim_group,
        index=group_df["trigger"],
        columns=group_df["trigger"]
    )

    display(df_sim_group)


GRUPO: ('nacer', 'embedded_proposition')


trigger,nacido,nació
trigger,,
nacido,1.000000,0.714802
nació,0.714802,1.000000


In [150]:
# ============================================================
# INTERPRETACIÓN INCREMENTAL DE NOVEDAD
# ============================================================

def classify_novelty(similarity):

    if similarity >= 0.85:
        return "redundant"

    elif similarity >= 0.60:
        return "variant"

    else:
        return "new_information"

In [151]:
# ============================================================
# CLASIFICAR SIMILITUD DENTRO DEL GRUPO
# ============================================================

sim_value = df_sim_group.loc["nacido", "nació"]

print(f"Similitud: {round(sim_value, 4)}")

print(
    "Clasificación:",
    classify_novelty(sim_value)
)

Similitud: 0.7148000001907349
Clasificación: variant


In [152]:
# ============================================================
# DETECCIÓN INCREMENTAL DE NOVEDAD RELACIONAL
# ============================================================

def compare_against_pool(candidate_row, seen_fragments):

    results = []

    candidate_text = candidate_row["fragment_no_entities"]

    for seen in seen_fragments:

        # Comparar solo si comparten lema y rol
        if (
            seen["lemma"] == candidate_row["lemma"]
            and
            seen["syntactic_role"] == candidate_row["syntactic_role"]
        ):

            emb = embedding_model.encode(
                [
                    candidate_text,
                    seen["fragment_no_entities"]
                ],
                normalize_embeddings=True
            )

            sim = cosine_similarity(
                [emb[0]],
                [emb[1]]
            )[0][0]

            results.append({
                "candidate_trigger": candidate_row["trigger"],
                "seen_trigger": seen["trigger"],
                "similarity": round(float(sim), 4)
            })

    return results

In [153]:
# ============================================================
# COMPARAR FRASE 1 CONTRA EL POOL ACTUAL
# ============================================================

comparison_results = []

for _, row in result_1["candidates"].iterrows():

    comparison_results.extend(
        compare_against_pool(row, seen_fragments)
    )

df_comparison = pd.DataFrame(comparison_results)

df_comparison

""


In [154]:
# ============================================================
# ACTUALIZAR POOL CON FRASE 1
# ============================================================

seen_entities.update(result_1["new_entities"])

for _, row in result_1["candidates"].iterrows():

    seen_fragments.append({
        "sentence_index": 1,
        "trigger": row["trigger"],
        "lemma": row["lemma"],
        "syntactic_role": row["syntactic_role"],
        "fragment": row["fragment_final"],
        "fragment_no_entities": row["fragment_no_entities"]
    })

print(f"Entidades vistas: {len(seen_entities)}")
print(f"Fragmentos vistos: {len(seen_fragments)}")

Entidades vistas: 7
Fragmentos vistos: 8


In [155]:
# ============================================================
# ESTADO ACTUAL DEL POOL
# ============================================================

print("="*120)
print("ENTIDADES")
print("="*120)

for ent in sorted(seen_entities):
    print("-", ent)

print("\n" + "="*120)
print("FRAGMENTOS")
print("="*120)

df_pool = pd.DataFrame(seen_fragments)

display(df_pool[[
    "sentence_index",
    "trigger",
    "lemma",
    "syntactic_role",
    "fragment_no_entities"
]])

ENTIDADES
- Cosme Pérez
- Cotarelo
- Genealogía
- Juan Rana
- Madrid
- Tudela de Duero
- Valladolid

FRAGMENTOS


,sentence_index,trigger,lemma,syntactic_role,fragment_no_entities
0,0,conocido,conocer,NaN,NaN
1,0,documenta,documentar,NaN,NaN
2,0,aplicaba,aplicar,NaN,NaN
3,1,aventuró,aventurar,subordinate,Aunque [PER] aventuró que pudo haber nacido en...
4,1,nacido,nacer,embedded_proposition,que pudo haber nacido en [LOC]
5,1,sabemos,saber,main,"que pudo haber nacido en [LOC], hoy sabemos qu..."
6,1,nació,nacer,embedded_proposition,"en [LOC], hoy sabemos que [PER] nació en [LOC]..."
7,1,bautizado,bautizar,subordinate,en cuya iglesia parroquial [PER] fue bautizado...


In [156]:
# ============================================================
# PROCESAR FRASE 2
# ============================================================

result_2 = process_incremental_sentence(2)

result_2["candidates"] = result_2["candidates"].copy()

if len(result_2["candidates"]) > 0:

    result_2["candidates"]["fragment_final"] = (
        result_2["candidates"].apply(select_fragment, axis=1)
    )

    result_2["candidates"]["fragment_no_entities"] = (
        result_2["candidates"]["fragment_final"].apply(remove_entities)
    )

    result_2["candidates"]["syntactic_role"] = (
        result_2["candidates"]["dep"].apply(syntactic_role)
    )

print("="*120)
print("FRASE")
print("="*120)
print(result_2["sentence"])

print("\n" + "="*120)
print("ENTIDADES NUEVAS")
print("="*120)
print(result_2["new_entities"])

print("\n" + "="*120)
print("FRAGMENTOS")
print("="*120)

if len(result_2["candidates"]) > 0:
    display(result_2["candidates"][[
        "trigger",
        "lemma",
        "dep",
        "syntactic_role",
        "fragment_no_entities"
    ]])
else:
    print("No se han extraído fragmentos relacionales en esta frase.")

FRASE
Cosme Pérez fue hijo de Damián Pérez e Isabel de Basto, su segunda esposa, y fue el cuarto de una familia de cuatro hermanos.

ENTIDADES NUEVAS
{'Damián Pérez', 'Isabel de Basto'}

FRAGMENTOS
No se han extraído fragmentos relacionales en esta frase.


In [157]:
# ============================================================
# DETECCIÓN DE ESTRUCTURAS COPULATIVAS
# ============================================================

def extract_copulative_candidates(sentence):

    doc = nlp(sentence)

    rows = []

    for token in doc:

        # Buscar atributos ligados a "ser"
        if token.dep_ == "attr":

            head = token.head

            if head.lemma_ == "ser":

                fragment = " ".join([
                    t.text for t in head.subtree
                ])

                rows.append({
                    "trigger": head.text,
                    "lemma": head.lemma_,
                    "dep": head.dep_,
                    "attribute": token.text,
                    "fragment": clean_fragment(fragment)
                })

    return pd.DataFrame(rows)

In [158]:
# ============================================================
# PROBAR COPULATIVAS EN FRASE 2
# ============================================================

df_cop_2 = extract_copulative_candidates(
    df_sentences.loc[2, "sentence"]
)

df_cop_2

""


In [159]:
# ============================================================
# INSPECCIÓN SINTÁCTICA - FRASE 2
# ============================================================

sentence = df_sentences.loc[2, "sentence"]
doc = nlp(sentence)

print(sentence)

for token in doc:
    print(
        f"{token.text:20} lemma={token.lemma_:15} "
        f"pos={token.pos_:8} dep={token.dep_:12} head={token.head.text}"
    )

Cosme Pérez fue hijo de Damián Pérez e Isabel de Basto, su segunda esposa, y fue el cuarto de una familia de cuatro hermanos.
Cosme                lemma=Cosme           pos=PROPN    dep=nsubj        head=hijo
Pérez                lemma=Pérez           pos=PROPN    dep=flat         head=Cosme
fue                  lemma=ser             pos=AUX      dep=cop          head=hijo
hijo                 lemma=hijo            pos=NOUN     dep=ROOT         head=hijo
de                   lemma=de              pos=ADP      dep=case         head=Damián
Damián               lemma=Damián          pos=PROPN    dep=nmod         head=hijo
Pérez                lemma=Pérez           pos=PROPN    dep=flat         head=Damián
e                    lemma=e               pos=CCONJ    dep=cc           head=Isabel
Isabel               lemma=Isabel          pos=PROPN    dep=conj         head=Damián
de                   lemma=de              pos=ADP      dep=case         head=Basto
Basto                lemma=Basto  

In [160]:
# ============================================================
# DETECCIÓN DE PREDICADOS NOMINALES CON CÓPULA
# ============================================================

def extract_nominal_predicates(sentence):

    doc = nlp(sentence)
    rows = []

    for token in doc:

        has_copula = any(
            child.dep_ == "cop" and child.lemma_ == "ser"
            for child in token.children
        )

        if has_copula and token.pos_ in ["NOUN", "ADJ"]:

            fragment = " ".join([t.text for t in token.subtree])

            rows.append({
                "trigger": token.text,
                "lemma": token.lemma_,
                "dep": token.dep_,
                "fragment": clean_fragment(fragment)
            })

    return pd.DataFrame(rows)

In [161]:
df_nominal_2 = extract_nominal_predicates(
    df_sentences.loc[2, "sentence"]
)

df_nominal_2

,trigger,lemma,dep,fragment
0,hijo,hijo,ROOT,Cosme Pérez fue hijo de Damián Pérez e Isabel ...
1,cuarto,cuarto,conj,y fue el cuarto de una familia de cuatro hermanos


In [162]:
# ============================================================
# UNIFICAR FRAGMENTOS VERBALES Y NOMINALES - FRASE 2
# ============================================================

df_verbal_2 = result_2["candidates"].copy()

df_verbal_2["source_type"] = "verbal"
df_nominal_2["source_type"] = "nominal"

df_nominal_2 = df_nominal_2.rename(columns={
    "fragment": "fragment_final"
})

df_nominal_2["syntactic_role"] = df_nominal_2["dep"].apply(syntactic_role)
df_nominal_2["fragment_no_entities"] = df_nominal_2["fragment_final"].apply(remove_entities)

df_candidates_2_all = pd.concat(
    [
        df_verbal_2,
        df_nominal_2
    ],
    ignore_index=True,
    sort=False
)

df_candidates_2_all[[
    "source_type",
    "trigger",
    "lemma",
    "dep",
    "syntactic_role",
    "fragment_no_entities"
]]

,source_type,trigger,lemma,dep,syntactic_role,fragment_no_entities
0,nominal,hijo,hijo,ROOT,main,"[PER] fue hijo de [PER] e [PER] , su segunda e..."
1,nominal,cuarto,cuarto,conj,coordinated,y fue el cuarto de una familia de cuatro hermanos


In [163]:
# ============================================================
# COMPARAR FRASE 2 CONTRA EL POOL ACTUAL
# ============================================================

comparison_results_2 = []

for _, row in df_candidates_2_all.iterrows():

    comparison_results_2.extend(
        compare_against_pool(row, seen_fragments)
    )

df_comparison_2 = pd.DataFrame(comparison_results_2)

df_comparison_2

""


In [164]:
# ============================================================
# ACTUALIZAR POOL CON FRASE 2
# ============================================================

seen_entities.update(result_2["new_entities"])

for _, row in df_candidates_2_all.iterrows():

    seen_fragments.append({
        "sentence_index": 2,
        "trigger": row["trigger"],
        "lemma": row["lemma"],
        "syntactic_role": row["syntactic_role"],
        "fragment": row["fragment_final"],
        "fragment_no_entities": row["fragment_no_entities"],
        "source_type": row["source_type"]
    })

print(f"Entidades vistas: {len(seen_entities)}")
print(f"Fragmentos vistos: {len(seen_fragments)}")

Entidades vistas: 9
Fragmentos vistos: 10


In [165]:
# ============================================================
# ESTADO ACTUAL DEL POOL
# ============================================================

print("="*120)
print("ENTIDADES")
print("="*120)

for ent in sorted(seen_entities):
    print("-", ent)

print("\n" + "="*120)
print("FRAGMENTOS")
print("="*120)

df_pool = pd.DataFrame(seen_fragments)

display(df_pool[[
    "sentence_index",
    "trigger",
    "lemma",
    "syntactic_role",
    "fragment_no_entities"
]])

ENTIDADES
- Cosme Pérez
- Cotarelo
- Damián Pérez
- Genealogía
- Isabel de Basto
- Juan Rana
- Madrid
- Tudela de Duero
- Valladolid

FRAGMENTOS


,sentence_index,trigger,lemma,syntactic_role,fragment_no_entities
0,0,conocido,conocer,NaN,NaN
1,0,documenta,documentar,NaN,NaN
2,0,aplicaba,aplicar,NaN,NaN
3,1,aventuró,aventurar,subordinate,Aunque [PER] aventuró que pudo haber nacido en...
4,1,nacido,nacer,embedded_proposition,que pudo haber nacido en [LOC]
5,1,sabemos,saber,main,"que pudo haber nacido en [LOC], hoy sabemos qu..."
6,1,nació,nacer,embedded_proposition,"en [LOC], hoy sabemos que [PER] nació en [LOC]..."
7,1,bautizado,bautizar,subordinate,en cuya iglesia parroquial [PER] fue bautizado...
8,2,hijo,hijo,main,"[PER] fue hijo de [PER] e [PER] , su segunda e..."
9,2,cuarto,cuarto,coordinated,y fue el cuarto de una familia de cuatro hermanos
